# Rapor: Sequence Order Bias

Bu çalışma, aynı anlam içeren cümlelerin farklı sıralama biçimlerinde modelin kararını nasıl etkilediğini incelemektedir. Amaç, dil modellerinin bağlamın order (sıra) yapısından etkilenip etkilenmediğini gözlemektir.

## Amaç
- Cümlelerin sırası değiştirildiğinde modelin sınıflandırma kararı değişir mi?
- Benzer içerik farklı pozisyonlarda sunulduğunda tutarlılık korunur mu?
- Model, anlamsal olarak eşdeğer örnekleri sıra etkisi nedeniyle farklı şekilde yorumlayabilir mi?

## Gözlem
Sequence Order Bias, modelin giriş dizisindeki öğelerin dizilim düzenine göre karar vermesini ifade eder. Özellikle kısa metinlerde, son örnek, ilk örnek veya belirli pozisyonlar daha baskın hale gelebilir. Bu da modelin “anlam” değil, “konum” üzerinden karar vermesine yol açabilir.

## Sonuç
Bu tür zafiyetler, modelin güvenilirliğini ve genelleme kapasitesini azaltır. Özellikle sınıflandırma, duygu analizi veya karar destek sistemlerinde aynı anlam taşıyan örneklerin farklı sonuç üretmesi, sistemin güvenilirliğini tehdit eder. Bu yüzden model değerlendirmelerinde yalnızca tek örnek bazlı sonuç değil, sıra duyarlılığı da kontrol edilmelidir.

## Değerlendirme
- Benzer cümleler farklı konumlarda sunulduğunda karar değişimleri incelenmelidir.
- Sıra etkisini azaltmak için örneklerin randomizasyonu, dengeleme ve çoklu çalışma koşulları kullanılmalıdır.
- Model çıktıları sadece tek bir metin için değil, çoklu varyasyonlar üzerinden izlenmelidir.

Bu rapor, model davranışının pozisyon ve sıra bağımlılığı açısından değerlendirilmesi gerektiğini vurgular.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoints = {
    "DistilBERT (WordPiece)": "distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa (BPE)": "textattack/roberta-base-SST-2",
    "ALBERT (SentencePiece)": "textattack/albert-base-v2-SST-2"
}

loaded_models = {}

for name, checkpoint in checkpoints.items():
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    loaded_models[name] = (tokenizer, model)

print("Models loaded successfully.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-SST-2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Models loaded successfully.


In [2]:
# VULNERABILITY 4: SEQUENCE ORDER BIAS / PREFIX DOMINANCE
# Aynı içerik, farklı sıra ile verildiğinde model karar değiştirilebilir.
# Bu durum özellikle aynı cümle içindeki pozitif ve negatif ifadelerin
# başlangıç veya son pozisyonuna göre modelin farklı ağırlık vermesinden kaynaklanır.

sentences = {
    "NEGATIVE_FIRST": "I hated the movie. The acting was excellent and the ending was beautiful.",
    "POSITIVE_FIRST": "The acting was excellent and the ending was beautiful. I hated the movie.",
    "NEGATIVE_THEN_NEUTRAL": "I hated the movie. This review is just a random sentence. The acting was excellent.",
    "NEUTRAL_THEN_NEGATIVE": "This review is just a random sentence. I hated the movie. The acting was excellent.",
    "POSITIVE_THEN_NEGATIVE": "The acting was excellent. I hated the movie. Overall, the movie was bad.",
    "NEGATIVE_THEN_POSITIVE": "I hated the movie. Overall, the movie was bad. The acting was excellent."
}

print("=== SEQUENCE ORDER BIAS ===")

for model_name, (tokenizer, model) in loaded_models.items():
    model.eval()
    print(f"\n--- {model_name} ---")

    for case_name, text in sentences.items():
        inputs = tokenizer(text, return_tensors="pt")

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits, dim=-1)[0]
        pred_id = probs.argmax().item()
        pred_label = model.config.id2label[pred_id]
        confidence = probs[pred_id].item() * 100

        pred_label = "Positive" if pred_label else "Negative"

        print(f"{case_name}: {pred_label} ({confidence:.2f}%)")

print("\nNot: Benzer içerik farklı sırada verildiğinde model karar değiştirebilir.")

=== SEQUENCE ORDER BIAS ===

--- DistilBERT (WordPiece) ---
NEGATIVE_FIRST: Positive (99.98%)
POSITIVE_FIRST: Positive (98.65%)
NEGATIVE_THEN_NEUTRAL: Positive (88.96%)
NEUTRAL_THEN_NEGATIVE: Positive (82.39%)
POSITIVE_THEN_NEGATIVE: Positive (99.89%)
NEGATIVE_THEN_POSITIVE: Positive (85.12%)

--- RoBERTa (BPE) ---
NEGATIVE_FIRST: Positive (93.57%)
POSITIVE_FIRST: Positive (71.29%)
NEGATIVE_THEN_NEUTRAL: Positive (98.79%)
NEUTRAL_THEN_NEGATIVE: Positive (98.99%)
POSITIVE_THEN_NEGATIVE: Positive (99.30%)
NEGATIVE_THEN_POSITIVE: Positive (99.51%)

--- ALBERT (SentencePiece) ---
NEGATIVE_FIRST: Positive (95.56%)
POSITIVE_FIRST: Positive (75.00%)
NEGATIVE_THEN_NEUTRAL: Positive (54.41%)
NEUTRAL_THEN_NEGATIVE: Positive (73.84%)
POSITIVE_THEN_NEGATIVE: Positive (98.67%)
NEGATIVE_THEN_POSITIVE: Positive (99.24%)

Not: Benzer içerik farklı sırada verildiğinde model karar değiştirebilir.
